# AF2·06 — Invariant Point Attention (IPA)

**Mechanism of the day:** attention that reasons about points in 3D and is *provably*
unchanged when you rotate or translate the whole protein. This is the engine of the
structure module, and the single most ingenious module in AlphaFold.

The problem it solves: we want residues to attend to each other using **geometry** — how
far apart they are, how they are oriented. But geometry described in absolute coordinates
is a trap. If your attention depends on raw `(x, y, z)`, then rotating the input protein
changes every number, and the network has to *learn* to be invariant from data — wasting
capacity and never getting it exactly right. AlphaFold instead makes invariance a
**property of the architecture**: it cannot be broken, because of how the math is built.

The trick uses the residue frames from rung 05. Each residue emits **query, key, and
value points in its own local frame**. To compare them, IPA:

1. maps every point into global space through its residue's frame (`to_global`),
2. builds attention logits from **global squared distances** between query points (of
   `i`) and key points (of `j`) — plus the usual scalar attention and a pair bias,
3. aggregates value points in global space, then maps the result **back into each
   residue's local frame** (`to_local`).

Steps 1 and 3 are inverses that both ride along with any global motion, so it cancels
out. Distances in step 2 are invariant to global motion to begin with. The result: the
per-residue update IPA produces is identical no matter how the protein is posed. You
will build the three geometric pieces, assemble them, and **measure** the invariance —
then watch a naive coordinate-based attention fail the same test spectacularly.

**How to use this notebook:** implement the reps, make the checkpoints pass. Solutions at
the bottom. No training — runs in seconds.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams['axes.spines.top'] = False; plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK, RED = '#2a78d6', '#008300', '#52514e', '#e34948'

L = 14                       # residues
c = 32                       # single-representation channels
cz = 16                      # pair-representation channels
h = 4                        # attention heads
d = 8                        # scalar query/key dim per head
Np = 4                       # query/key POINTS per head
Npv = 6                      # value POINTS per head

def rand_rotations(n):
    A = torch.randn(n, 3, 3); Q, S = torch.linalg.qr(A)
    Q = Q * torch.sign(torch.diagonal(S, dim1=-2, dim2=-1))[:, None, :]
    Q[torch.det(Q) < 0, :, 0] *= -1
    return Q

# a toy structure to attend over: frames + single rep + pair rep
Rs = rand_rotations(L); ts = torch.cumsum(torch.randn(L, 3), 0)
s_rep = torch.randn(L, c); z_rep = torch.randn(L, L, cz)
print('IPA over L=%d residues | %d heads, %d query/key points, %d value points per head'
      % (L, h, Np, Npv))

## Part 1 — points into global space

Each residue produces `Np` query points and `Np` key points, expressed **in its own
local frame**. To compare a query point of residue `i` with a key point of residue `j`,
they must live in the same space — so we push both into global coordinates with the
residue frames (exactly `to_global` from rung 05, batched over heads and points).

### Rep 1 — `points_to_global(Rs, ts, local_points)`
`local_points` is `[L, h, P, 3]` (per residue, head, point). Apply each residue's frame:
`global = R · local + t`. Return `[L, h, P, 3]` in global coordinates.

In [ ]:
def points_to_global(Rs, ts, local_points):
    '''Place per-residue local points into global space via their frames.'''
    # YOUR CODE HERE
    # hint: einsum('lij,lhpj->lhpi', Rs, local_points) + ts[:, None, None, :]
    raise NotImplementedError

# --- checkpoint ---
lp = torch.randn(L, h, Np, 3)
g = points_to_global(Rs, ts, lp)
assert g.shape == (L, h, Np, 3)
# residue 0's points, placed manually, must match
assert torch.allclose(g[0, 1, 2], Rs[0] @ lp[0, 1, 2] + ts[0], atol=1e-5)
# a point at the local origin lands on the residue's CA (its translation)
z_origin = torch.zeros(L, h, Np, 3)
assert torch.allclose(points_to_global(Rs, ts, z_origin)[:, 0, 0, :], ts, atol=1e-5)
print('points_to_global ok — local points placed into the structure ✓')

## Part 2 — the invariant attention logit

The IPA logit for `(i, j)`, per head, sums three terms:

$$
\mathrm{logit}[i,j] = \frac{q_s[i]\cdot k_s[j]}{\sqrt{d}} + \mathrm{bias}(z[i,j]) - \frac{\gamma}{2}\sum_{\text{points}} \big\lVert Q[i] - K[j]\big\rVert^2
$$

The first two are the attention you already know. The **point term** is the new,
geometry-aware ingredient — and it is invariant for a beautiful reason: it depends only
on **distances between global points**, and distances do not change when you move the
whole protein. Residues whose query/key points land near each other in space get a high
logit; a global rotation moves both sets of points together, so the distance — and the
logit — is untouched.

`gamma` is a per-head learnable weight (kept positive) that lets each head decide how
much to trust geometry versus the scalar/pair terms.

### Rep 2 — `point_distance_term(Qg, Kg)`
`Qg, Kg` are `[L, h, Np, 3]` global query/key points. Return `[L, L, h]`: for each pair
`(i, j)` and head, the **sum over points** of squared distance
`||Qg[i, ·, p] − Kg[j, ·, p]||²`. (The `−gamma/2` scaling is applied later.)

In [ ]:
def point_distance_term(Qg, Kg):
    '''Sum of squared distances between residue i query points and residue j key points -> [L,L,h].'''
    # YOUR CODE HERE
    # hint: diff = Qg[:, None] - Kg[None]      # [L, L, h, Np, 3]
    #       return (diff ** 2).sum(-1).sum(-1) # sum over the 3 coords, then over points
    raise NotImplementedError

# --- checkpoint ---
Qg = points_to_global(Rs, ts, torch.randn(L, h, Np, 3))
Kg = points_to_global(Rs, ts, torch.randn(L, h, Np, 3))
dt = point_distance_term(Qg, Kg)
assert dt.shape == (L, L, h) and (dt >= 0).all(), 'squared distances are non-negative'
# THE property: move the whole structure -> this term is unchanged
Rg = rand_rotations(1)[0]; tg = torch.randn(3)
Qg2 = Qg @ Rg.T + tg; Kg2 = Kg @ Rg.T + tg
assert torch.allclose(dt, point_distance_term(Qg2, Kg2), atol=1e-4), 'point term must be pose-invariant'
print('point term ok — built from global distances, unchanged by a global move ✓')

## Part 3 — aggregating value points, and coming home to local frames

The output side mirrors the input side. Each residue also emits `Npv` **value points** in
its local frame. IPA pushes them to global space, takes the attention-weighted average
(so residue `i` collects value points from the residues it attends to), and then — the
key step — maps the aggregated points **back into residue `i`'s own local frame** with
`to_local`.

That final pull-back is what closes the invariance loop: the value points went out
through the residues' frames and come back through residue `i`'s frame, so any global
motion applied in between cancels exactly. The returned points are expressed in `i`'s
local view, ready to become part of its updated representation.

### Rep 3 — `aggregate_value_points(a, Vg, Rs, ts)`
`a` is the attention `[L, L, h]` (already softmaxed over `j`), `Vg` the global value
points `[L, h, Npv, 3]`. Aggregate `sum_j a[i,j] · Vg[j]` and express the result in each
residue `i`'s local frame. Return `[L, h, Npv, 3]`.

In [ ]:
def aggregate_value_points(a, Vg, Rs, ts):
    '''Attention-weighted global value points, mapped back into each residue local frame.'''
    # YOUR CODE HERE
    # hint: global_out = einsum('ijh,jhpx->ihpx', a, Vg)          # weighted sum over j
    #       local_out  = einsum('lji,lhpj->lhpi', Rs, global_out - ts[:, None, None, :])
    #       (that einsum applies R^T, i.e. to_local, per residue)
    raise NotImplementedError

# --- checkpoint ---
a = torch.softmax(torch.randn(L, L, h), dim=1)
Vg = points_to_global(Rs, ts, torch.randn(L, h, Npv, 3))
loc = aggregate_value_points(a, Vg, Rs, ts)
assert loc.shape == (L, h, Npv, 3)
# invariance: aggregate under a moved structure, in local frames it is identical
Vg2 = Vg @ Rg.T + tg
Rs2 = torch.einsum('ij,njk->nik', Rg, Rs); ts2 = ts @ Rg.T + tg
loc2 = aggregate_value_points(a, Vg2, Rs2, ts2)
assert torch.allclose(loc, loc2, atol=1e-4), 'aggregated value points must be pose-invariant in local frames'
print('value aggregation ok — points go out through frames and come home invariant ✓')

## Part 4 — assemble IPA, and measure the invariance

Wire the pieces into the full module (projections given). The forward pass builds the
three-term logit, softmaxes over `j`, and gathers three kinds of output — scalar values,
pair values, and your aggregated value points (plus their norms) — into the per-residue
update.

Then the headline test: run IPA on a structure and on a **globally rotated + translated**
copy, and compare the updates. They should be identical to floating-point precision —
not because we trained for it, but because the architecture cannot do otherwise.

In [ ]:
class IPA(nn.Module):
    def __init__(self):
        super().__init__()
        self.qs = nn.Linear(c, h * d); self.ks = nn.Linear(c, h * d); self.vs = nn.Linear(c, h * d)
        self.qp = nn.Linear(c, h * Np * 3); self.kp = nn.Linear(c, h * Np * 3); self.vp = nn.Linear(c, h * Npv * 3)
        self.bz = nn.Linear(cz, h)
        self.gamma = nn.Parameter(torch.zeros(h))            # per-head geometry weight
        self.out = nn.Linear(h * d + h * cz + h * Npv * 3 + h * Npv, c)
    def forward(self, x, z, Rs, ts):
        qs = self.qs(x).view(L, h, d); ks = self.ks(x).view(L, h, d); vs = self.vs(x).view(L, h, d)
        Qg = points_to_global(Rs, ts, self.qp(x).view(L, h, Np, 3))
        Kg = points_to_global(Rs, ts, self.kp(x).view(L, h, Np, 3))
        Vg = points_to_global(Rs, ts, self.vp(x).view(L, h, Npv, 3))
        scal = torch.einsum('ihd,jhd->ijh', qs, ks) / math.sqrt(d)
        gamma = F.softplus(self.gamma)
        logit = scal + self.bz(z) - 0.5 * gamma[None, None] * point_distance_term(Qg, Kg)
        a = F.softmax(logit, dim=1)                          # attend over j
        o_scalar = torch.einsum('ijh,jhd->ihd', a, vs).reshape(L, h * d)
        o_pair = torch.einsum('ijh,ijz->ihz', a, z).reshape(L, h * cz)
        pts = aggregate_value_points(a, Vg, Rs, ts)
        o_points = pts.reshape(L, h * Npv * 3)
        o_norms = pts.norm(dim=-1).reshape(L, h * Npv)       # point norms are invariant scalars
        return self.out(torch.cat([o_scalar, o_pair, o_points, o_norms], -1))

ipa = IPA().eval()
with torch.no_grad():
    upd = ipa(s_rep, z_rep, Rs, ts)
    # a random global move
    Rg = rand_rotations(1)[0]; tg = torch.randn(3)
    Rs_moved = torch.einsum('ij,njk->nik', Rg, Rs); ts_moved = ts @ Rg.T + tg
    upd_moved = ipa(s_rep, z_rep, Rs_moved, ts_moved)
    # a genuinely DIFFERENT structure (to show the update is not just constant)
    upd_other = ipa(s_rep, z_rep, rand_rotations(L), torch.cumsum(torch.randn(L, 3), 0))

inv_err = (upd - upd_moved).abs().max().item()
geo_dep = (upd - upd_other).abs().max().item()
assert upd.shape == (L, c)
assert inv_err < 1e-4, 'IPA output must be invariant to a global move'
assert geo_dep > 0.1, 'but it must still depend on the actual geometry'
print('IPA update shape', tuple(upd.shape))
print('invariance error under a global rotation+translation: %.2e  (essentially zero)' % inv_err)
print('sensitivity to a genuinely different structure:        %.3f  (clearly nonzero) ✓' % geo_dep)

### Rep 4 — `naive_geometric_attention(x, ts, W)`
For contrast, the tempting wrong approach: attention that reads **absolute coordinates**.
Concatenate each residue's coordinate `ts[i]` onto its features, project with the given
`W` (`Linear(c+3, c)`), and do one step of plain self-attention (scaled dot-product,
softmax over `j`, weighted sum). This is a perfectly reasonable-looking geometric
attention — and it is **not** invariant, which is the whole point.

In [ ]:
def naive_geometric_attention(x, ts, W):
    '''Self-attention on features concatenated with ABSOLUTE coordinates (not invariant).'''
    # YOUR CODE HERE
    # hint: f = W(torch.cat([x, ts], -1))            # [L, c]
    #       att = softmax(f @ f.T / sqrt(c), dim=1)  # [L, L]
    #       return att @ f
    raise NotImplementedError

# --- checkpoint ---
W = nn.Linear(c + 3, c)
with torch.no_grad():
    n1 = naive_geometric_attention(s_rep, ts, W)
    n2 = naive_geometric_attention(s_rep, ts_moved, W)
naive_err = (n1 - n2).abs().max().item()
assert n1.shape == (L, c)
assert naive_err > 0.1, 'the naive version should visibly break under a global move'
print('naive coordinate attention change under the SAME global move: %.3f' % naive_err)
print('IPA changed by %.2e; the naive version by %.3f — ~%.0e times more. Architecture'
      % (inv_err, naive_err, naive_err / max(inv_err, 1e-12)))
print('beats training every time when a symmetry is exact. ✓')

fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.bar(['IPA\n(invariant)', 'naive coord\nattention'], [inv_err, naive_err], color=[BLUE, RED])
ax.set_yscale('log'); ax.set_ylabel('output change under a global move (log)')
ax.set_title('invariance: built in vs. hoped for'); ax.grid(alpha=.15, axis='y')
plt.show()

## Reflection — what just transferred

- **IPA is attention with a geometry term** that is invariant by construction. The logit
  adds, to the usual scalar attention and pair bias, a **negative squared-distance**
  between query and key *points* placed in global space through the residue frames.
- **Invariance comes from the round-trip.** Points go out through the residues' frames
  and the aggregated result comes back through the target residue's frame; global
  distances in between are already pose-invariant. Any global rotation/translation
  cancels — you measured `~1e-7`, at machine precision, with zero training.
- **This is why architecture beats data augmentation.** A naive attention on absolute
  coordinates changed by `O(1)` under the exact same move. Baking a symmetry into the
  math is exact and free; hoping a network learns it is neither.
- **Point norms are invariant scalars**, so IPA can safely feed them into the update
  alongside the invariant local-frame points.
- IPA is the *reader* of the structure module: it lets each residue gather geometric
  context from all the others. What it does not yet do is *move* anything — that is the
  next rung.

**Next rung:** `AF2·07 — FAPE loss & backbone generation`. IPA updates each residue's
representation; a small head turns that update into a **frame update** (a rotation +
translation that nudges the residue), we iterate to fold a backbone, and we train it with
**FAPE** — the frame-aligned point error, the loss that measures structural error in a
way as invariant as IPA itself.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def points_to_global(Rs, ts, local_points):
    return torch.einsum('lij,lhpj->lhpi', Rs, local_points) + ts[:, None, None, :]

def point_distance_term(Qg, Kg):
    diff = Qg[:, None] - Kg[None]                 # [L, L, h, Np, 3]
    return (diff ** 2).sum(-1).sum(-1)            # sum over coords, then over points

def aggregate_value_points(a, Vg, Rs, ts):
    global_out = torch.einsum('ijh,jhpx->ihpx', a, Vg)               # weighted sum over j
    return torch.einsum('lji,lhpj->lhpi', Rs, global_out - ts[:, None, None, :])   # to_local

def naive_geometric_attention(x, ts, W):
    f = W(torch.cat([x, ts], -1))
    att = torch.softmax(f @ f.T / math.sqrt(c), dim=1)
    return att @ f

print('reference solutions loaded — re-run the checkpoint cells above')